In [2]:
"""
Data exploration for Table 1 replication
把所有数据文件过一遍，判断是否需要 clean
"""

import pandas as pd
import numpy as np
from pathlib import Path



# ============================================================
# 1. crsp_sp500_daily.parquet
# ============================================================
print("=" * 60)
print("1. crsp_sp500_daily.parquet")
print("=" * 60)

df_spx = pd.read_parquet("../_data/crsp_sp500_daily.parquet")

print(f"\n shape: {df_spx.shape}")
print(f" date range: {df_spx['date'].min()} to {df_spx['date'].max()}")
print(f"\n columns:\n{df_spx.dtypes}")
print(f"\n missing values:\n{df_spx.isna().sum()}")
print(f"\n first 5 rows:")
print(df_spx.head())
print(f"\n last 5 rows:")
print(df_spx.tail())
print(f"\n basic stats:")
print(df_spx.describe())

# Check sprtrn range (should be small daily returns)
print(f"\n sprtrn (daily S&P return) range: {df_spx['sprtrn'].min():.4f} to {df_spx['sprtrn'].max():.4f}")
print(f" spindx range: {df_spx['spindx'].min():.2f} to {df_spx['spindx'].max():.2f}")

# ============================================================
# 2. crsp_treasury_returns.parquet
# ============================================================
print("\n" + "=" * 60)
print("2. crsp_treasury_returns.parquet")
print("=" * 60)

df_treas = pd.read_parquet("../_data/crsp_treasury_returns.parquet")

print(f"\n shape: {df_treas.shape}")
print(f" date range: {df_treas['date'].min()} to {df_treas['date'].max()}")
print(f"\n columns:\n{df_treas.dtypes}")
print(f"\n missing values:\n{df_treas.isna().sum()}")
print(f"\n first 5 rows:")
print(df_treas.head())
print(f"\n basic stats:")
print(df_treas.describe())

# Check if monthly or daily
date_diff = df_treas['date'].diff().dt.days.dropna()
print(f"\n avg days between obs: {date_diff.mean():.1f} (1=daily, ~21=monthly)")

# ============================================================
# 3. clean_options.parquet (已经clean过的)
# ============================================================
print("\n" + "=" * 60)
print("3. clean_options.parquet")
print("=" * 60)

df_opt = pd.read_parquet("../_data/clean_options.parquet")

print(f"\n shape: {df_opt.shape}")
print(f" date range: {df_opt['date'].min()} to {df_opt['date'].max()}")
print(f"\n columns:\n{df_opt.dtypes}")
print(f"\n missing values:\n{df_opt.isna().sum()}")
print(f"\n first 5 rows:")
print(df_opt.head())
print(f"\n moneyness range: {df_opt['moneyness'].min():.2f} to {df_opt['moneyness'].max():.2f}")
print(f" days_to_maturity range: {df_opt['days_to_maturity'].min()} to {df_opt['days_to_maturity'].max()}")
print(f" mid_price range: {df_opt['mid_price'].min():.2f} to {df_opt['mid_price'].max():.2f}")

# ============================================================
# 4. clean_rates.parquet (已经clean过的)
# ============================================================
print("\n" + "=" * 60)
print("4. clean_rates.parquet")
print("=" * 60)

df_rates = pd.read_parquet("../_data/clean_rates.parquet")

print(f"\n shape: {df_rates.shape}")
print(f" date range: {df_rates['date'].min()} to {df_rates['date'].max()}")
print(f"\n columns:\n{df_rates.dtypes}")
print(f"\n missing values:\n{df_rates.isna().sum()}")
print(f"\n first 5 rows:")
print(df_rates.head())
print(f"\n rf_1m range: {df_rates['rf_1m'].min():.6f} to {df_rates['rf_1m'].max():.6f}")

# ============================================================
# 5. 总结：需要做什么 clean？
# ============================================================
print("\n" + "=" * 60)
print("总结")
print("=" * 60)
print("""
crsp_sp500_daily：
  - 是否是日频？
  - 需要转成月频（取月末值）
  - 需要算月度 realized dividend

crsp_treasury_returns：
  - 是日频还是月频？
  - returns 是否已经是 log return？
  - 需要确认单位（小数还是百分比）

clean_options：已clean ✓
clean_rates：已clean ✓
""")

1. crsp_sp500_daily.parquet

 shape: (7300, 5)
 date range: 1996-01-02 00:00:00 to 2024-12-31 00:00:00

 columns:
date      datetime64[ns]
spindx           float64
sprtrn           float64
vwretd           float64
vwretx           float64
dtype: object

 missing values:
date      0
spindx    0
sprtrn    0
vwretd    0
vwretx    0
dtype: int64

 first 5 rows:
        date  spindx    sprtrn    vwretd    vwretx
0 1996-01-02  620.73  0.007793  0.006606  0.006598
1 1996-01-03  621.32  0.000950 -0.000155 -0.000291
2 1996-01-04  617.70 -0.005826 -0.007674 -0.007687
3 1996-01-05  616.72 -0.001587 -0.001023 -0.001029
4 1996-01-08  618.46  0.002821  0.002686  0.002463

 last 5 rows:
           date   spindx    sprtrn    vwretd    vwretx
7295 2024-12-24  6040.04  0.011043  0.010566  0.010521
7296 2024-12-26  6037.59 -0.000406  0.000346  0.000282
7297 2024-12-27  5970.84 -0.011056 -0.010692 -0.010775
7298 2024-12-30  5906.94 -0.010702 -0.009878 -0.009900
7299 2024-12-31  5881.63 -0.004285 -0.003392

In [ ]:
df_rates = pd.read_parquet("../_data/fama_french_monthly.parquet")

# 看一下不同时期的 rf_1m
print(df_rates[df_rates['date'].dt.year == 2000]['rf_1m_monthly'].describe())  # 高利率时期
print(df_rates[df_rates['date'].dt.year == 2007]['rf_1m_monthly'].describe())  # 金融危机前
print(df_rates[df_rates['date'].dt.year == 2020]['rf_1m_monthly'].describe())  # 近零利率

count    252.000000
mean       0.000225
std        0.000043
min        0.000200
25%        0.000200
50%        0.000200
75%        0.000200
max        0.000300
Name: rf_1m, dtype: float64
count    251.000000
mean       0.000183
std        0.000038
min        0.000100
25%        0.000200
50%        0.000200
75%        0.000200
max        0.000200
Name: rf_1m, dtype: float64
count    253.000000
mean       0.000025
std        0.000043
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        0.000100
Name: rf_1m, dtype: float64
